# Store Sales, Solution Achieves Top 23% on Public Leaderboard:
https://www.kaggle.com/competitions/store-sales-time-series-forecasting/overview

## Import Packages and Data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import seaborn as sns
import datetime as dt
import random
import lightgbm as lgbm
import optuna
from optuna.samplers import RandomSampler, TPESampler
from sklearn.metrics import f1_score, root_mean_squared_error
import itertools

In [2]:
DATA_DIR = "/Users/adamgolledge/Documents/PersonalProjects/Store Sales/Data"

train = pd.concat([
    pd.read_csv(f"{DATA_DIR}/train_chunk_1.csv"),
    pd.read_csv(f"{DATA_DIR}/train_chunk_2.csv"),
    pd.read_csv(f"{DATA_DIR}/train_chunk_3.csv"),
], ignore_index=True)

test = pd.read_csv(f"{DATA_DIR}/test.csv")
holidays = pd.read_csv(f"{DATA_DIR}/holidays_events.csv")
oil = pd.read_csv(f"{DATA_DIR}/oil.csv")
stores = pd.read_csv(f"{DATA_DIR}/stores.csv")
transactions = pd.read_csv(f"{DATA_DIR}/transactions.csv")

## Original Data Checking & Cleaning

In [3]:
og_datasets = [train, test]

In [4]:
# Check NAs in datasets

for set in og_datasets:
    print(set.isna().sum())

id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64
id             0
date           0
store_nbr      0
family         0
onpromotion    0
dtype: int64


In [5]:
# Check all families are equally represented (no mis-spelling)

for set in og_datasets:
    print(set["family"].value_counts().min())
    print(set["family"].value_counts().max())

90936
90936
864
864


In [6]:
# Check all store numbers are equally represented

for set in og_datasets:
    print(set["store_nbr"].value_counts().min())
    print(set["store_nbr"].value_counts().max())

55572
55572
528
528


In [7]:
# Check no missing dates

print(train["date"].min())
print(train["date"].max())

2013-01-01
2017-08-15


In [8]:
print((dt.datetime.strptime(train["date"].max(), '%Y-%m-%d') - dt.datetime.strptime(train["date"].min(), '%Y-%m-%d')).days + 1)
print(train["date"].nunique())

1688
1684


In [9]:
print((dt.datetime.strptime(test["date"].max(), '%Y-%m-%d') - dt.datetime.strptime(test["date"].min(), '%Y-%m-%d')).days + 1)
print(test["date"].nunique())

16
16


In [10]:
# No missings in test, 4 missings in train. Lets find them

all_dates = pd.date_range(
    start="2013-01-01",
    end="2017-08-15",
    freq="D"
).strftime("%Y-%m-%d").tolist()

train_dates = (train["date"].unique().tolist())

[i for i in all_dates if i not in train_dates]


['2013-12-25', '2014-12-25', '2015-12-25', '2016-12-25']

In [11]:
train.head(10)

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0
5,5,2013-01-01,1,BREAD/BAKERY,0.0,0
6,6,2013-01-01,1,CELEBRATION,0.0,0
7,7,2013-01-01,1,CLEANING,0.0,0
8,8,2013-01-01,1,DAIRY,0.0,0
9,9,2013-01-01,1,DELI,0.0,0


In [12]:
# Every Christmas day is missing. Stores are shut on Christmas so no chance of any sales.
# However, leaving out will ruin lagged features across Christmas, so putting them in is the way forward. Moving avg feats are then jeapordised but we can leave Christmas out of these.

train["date"] = pd.to_datetime(train["date"])
years = range(train["date"].dt.year.min(), train["date"].dt.year.max() + 1)
christmas_dates = pd.to_datetime([f"{y}-12-25" for y in years])
store_family = train[["store_nbr", "family"]].drop_duplicates()

christmas_rows = (
    store_family
    .assign(key=1)
    .merge(pd.DataFrame({"date": christmas_dates, "key": 1}), on="key")
    .drop("key", axis=1)
)

existing_keys = train[["store_nbr", "family", "date"]]

christmas_rows = (
    christmas_rows
    .merge(existing_keys, on=["store_nbr", "family", "date"], how="left", indicator=True)
    .query("_merge == 'left_only'")
    .drop("_merge", axis=1)
)

christmas_rows["sales"] = 0.0
christmas_rows["onpromotion"] = 0

train = pd.concat([train, christmas_rows], ignore_index=True)
train = train.sort_values(["date", "store_nbr", "family"]).reset_index(drop=True)

train["id"] = range(len(train))

train = train[train["date"]!="2017-12-25"].reset_index(drop=True)

In [13]:
# Find no-sale dates
 
no_sale_dates = train.groupby("date").agg({"sales": "sum"}).reset_index()
no_sale_dates.columns = ["date", "total_sales"]
no_sale_dates["total_sales"] = no_sale_dates["total_sales"]/1000
no_sale_dates.sort_values("total_sales").head(20)

,date,total_sales
723,2014-12-25,0.000000
1088,2015-12-25,0.000000
358,2013-12-25,0.000000
1454,2016-12-25,0.000000
0,2013-01-01,2.511619
365,2014-01-01,8.602065
1461,2017-01-01,12.082501
730,2015-01-01,12.773617
1095,2016-01-01,16.433394
23,2013-01-24,247.245691


In [14]:
# Make day of week and month of year

og_datasets = [train, test]

for set in og_datasets:
    set["date"] = pd.to_datetime(set["date"])
    set["day_of_week"] = set["date"].dt.dayofweek
    set["day_of_month"] = set["date"].dt.day
    set["week_of_year"] = set["date"].dt.isocalendar().week.astype(int)
    set["month_of_year"] = set["date"].dt.month
    set["christmas_flag"] = np.where((set["day_of_month"]==25) & (set["month_of_year"]==12), 1, 0)
    set["nyd_flag"] = np.where((set["day_of_month"]==1) & (set["month_of_year"]==1), 1, 0)

## Auxiliary Data Cleaning and Joining

### Store Detail and Holiday Data

In [15]:
# Group hols by date, locale, and locale name to see if any duplicates

holiday_counts = (
    holidays
    .groupby(["date", "locale", "locale_name"])
    .size()
    .reset_index(name="count")
)

holiday_counts[holiday_counts["count"] > 1]

affected_dates = holiday_counts[holiday_counts["count"] > 1]["date"].unique().tolist()

In [16]:
# There are dupes, check what they are

holidays[holidays["date"].isin(affected_dates)]

,date,type,locale,locale_name,description,transferred
35,2012-12-24,Bridge,National,Ecuador,Puente Navidad,False
36,2012-12-24,Additional,National,Ecuador,Navidad-1,False
39,2012-12-31,Bridge,National,Ecuador,Puente Primer dia del ano,False
40,2012-12-31,Additional,National,Ecuador,Primer dia del ano-1,False
156,2014-12-26,Bridge,National,Ecuador,Puente Navidad,False
157,2014-12-26,Additional,National,Ecuador,Navidad+1,False
235,2016-05-01,Holiday,National,Ecuador,Dia del Trabajo,False
236,2016-05-01,Event,National,Ecuador,Terremoto Manabi+15,False
242,2016-05-07,Additional,National,Ecuador,Dia de la Madre-1,False
243,2016-05-07,Event,National,Ecuador,Terremoto Manabi+21,False


In [17]:
# -1s (pre-holidays) are causing some issues here. Terremonto Manabi are also causing issues (the bad earthquakes of 2016). Lets get lists of these

unique_hols = list(holidays["description"].unique())

pre_hols = [i for i in unique_hols if "-" in i and "futbol" not in i]
post_hols = [i for i in unique_hols if "+" in i]

In [18]:
pre_hols

['Fundacion de Quito-1',
 'Navidad-4',
 'Navidad-3',
 'Navidad-2',
 'Navidad-1',
 'Primer dia del ano-1',
 'Dia de la Madre-1',
 'Fundacion de Guayaquil-1']

In [19]:
post_hols

['Navidad+1',
 'Terremoto Manabi+1',
 'Terremoto Manabi+2',
 'Terremoto Manabi+3',
 'Terremoto Manabi+4',
 'Terremoto Manabi+5',
 'Terremoto Manabi+6',
 'Terremoto Manabi+7',
 'Terremoto Manabi+8',
 'Terremoto Manabi+9',
 'Terremoto Manabi+10',
 'Terremoto Manabi+11',
 'Terremoto Manabi+12',
 'Terremoto Manabi+13',
 'Terremoto Manabi+14',
 'Terremoto Manabi+15',
 'Terremoto Manabi+16',
 'Terremoto Manabi+17',
 'Terremoto Manabi+18',
 'Terremoto Manabi+19',
 'Terremoto Manabi+20',
 'Terremoto Manabi+21',
 'Terremoto Manabi+22',
 'Terremoto Manabi+23',
 'Terremoto Manabi+24',
 'Terremoto Manabi+25',
 'Terremoto Manabi+26',
 'Terremoto Manabi+27',
 'Terremoto Manabi+28',
 'Terremoto Manabi+29',
 'Terremoto Manabi+30']

In [20]:
# From the data description, transferred holidays are those that are moved to a different date (usually as the calendar date falls on a weekend)
# The original holiday should be ingnored in favour of the transferred one. Lets see which these are

holidays[holidays["transferred"]==True]

,date,type,locale,locale_name,description,transferred
19,2012-10-09,Holiday,National,Ecuador,Independencia de Guayaquil,True
72,2013-10-09,Holiday,National,Ecuador,Independencia de Guayaquil,True
135,2014-10-09,Holiday,National,Ecuador,Independencia de Guayaquil,True
255,2016-05-24,Holiday,National,Ecuador,Batalla de Pichincha,True
266,2016-07-25,Holiday,Local,Guayaquil,Fundacion de Guayaquil,True
268,2016-08-10,Holiday,National,Ecuador,Primer Grito de Independencia,True
297,2017-01-01,Holiday,National,Ecuador,Primer dia del ano,True
303,2017-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,True
312,2017-05-24,Holiday,National,Ecuador,Batalla de Pichincha,True
324,2017-08-10,Holiday,National,Ecuador,Primer Grito de Independencia,True


In [21]:
# Write drop_target in two parts, one to show anything transferred, another to show pre/post holidays

holidays["drop_target"] = np.where(holidays["transferred"]==True, 1, 0)
holidays["drop_target"] = np.where(holidays["type"]=="Work Day", 1, holidays["drop_target"])
holidays["drop_target"] = np.where(holidays["description"].isin(pre_hols+post_hols), 1, holidays["drop_target"])

holidays = holidays[holidays["drop_target"]==0]

In [22]:
holidays["date"] = pd.to_datetime(holidays["date"])

nat_cal = holidays.loc[holidays["locale"] == "National", ["date", "description", "locale_name"]].sort_values("date")
reg_cal = holidays.loc[holidays["locale"] == "Regional", ["date", "description", "locale_name"]].sort_values("date")
loc_cal = holidays.loc[holidays["locale"] == "Local", ["date", "description", "locale_name"]].sort_values("date")

grans = ["national", "regional", "local"]

gran_md = {
    "national": nat_cal,
    "regional": reg_cal,
    "local": loc_cal
}

col_md = {
    "national": None,
    "regional": "state",
    "local": "city"
}

In [23]:
store_cols = [c for c in stores.columns if c != "store_nbr"]
if any(col in train.columns for col in store_cols) or any(col in test.columns for col in store_cols):
    raise RuntimeError("Stores already merged into train/test — aborting to avoid duplicate columns.")
train = train.merge(stores, how="left", on="store_nbr")
test = test.merge(stores, how="left", on="store_nbr")

In [24]:
# National hols to train

train["date"] = pd.to_datetime(train["date"])
nat_cal["date"] = pd.to_datetime(nat_cal["date"])

nat_cal = nat_cal.sort_values("date")

nat_cal_renamed = nat_cal.rename(
    columns={
        "date": "national_holiday_date",
        "description": "next_national_holiday",
        "locale_name": "country"
    }
)

train = pd.merge_asof(
    train,
    nat_cal_renamed,
    left_on="date",
    right_on="national_holiday_date",
    direction="forward"
)

train["days_until_next_national_holiday"] = (
    train["national_holiday_date"] - train["date"]
).dt.days

train = train.drop(["national_holiday_date"], axis=1)

train["next_national_holiday"] = np.where(train["days_until_next_national_holiday"]>21, np.nan, train["next_national_holiday"])
train["days_until_next_national_holiday"] = np.where(train["days_until_next_national_holiday"]>21, np.nan, train["days_until_next_national_holiday"])

train["binned_days_until_next_national_holiday"] = np.where(train["days_until_next_national_holiday"]<2, 1, 
                                                            np.where(train["days_until_next_national_holiday"]<8, 2, 
                                                                     np.where(train["days_until_next_national_holiday"]<=21, 3, 4)))

train = train.drop(["days_until_next_national_holiday", "country"], axis=1)

In [25]:
# Regional hols to train

train["date"] = pd.to_datetime(train["date"])
reg_cal["date"] = pd.to_datetime(reg_cal["date"])

reg_cal = reg_cal.sort_values("date")

reg_cal_renamed = reg_cal.rename(
    columns={
        "date": "regional_holiday_date",
        "description": "next_regional_holiday",
        "locale_name": "state"
    }
)

train = pd.merge_asof(
    train,
    reg_cal_renamed,
    left_on="date",
    right_on="regional_holiday_date",
    by="state",
    direction="forward"
)

train["days_until_next_regional_holiday"] = (
    train["regional_holiday_date"] - train["date"]
).dt.days

train = train.drop(["regional_holiday_date"], axis=1)

train["next_regional_holiday"] = np.where(train["days_until_next_regional_holiday"]>21, np.nan, train["next_regional_holiday"])
train["days_until_next_regional_holiday"] = np.where(train["days_until_next_regional_holiday"]>21, np.nan, train["days_until_next_regional_holiday"])

train["binned_days_until_next_regional_holiday"] = np.where(train["days_until_next_regional_holiday"]<2, 1, 
                                                            np.where(train["days_until_next_regional_holiday"]<8, 2, 
                                                                     np.where(train["days_until_next_regional_holiday"]<=21, 3, 4)))

train = train.drop("days_until_next_regional_holiday", axis=1)

In [26]:
# Local hols to train

train["date"] = pd.to_datetime(train["date"])
loc_cal["date"] = pd.to_datetime(loc_cal["date"])

loc_cal = loc_cal.sort_values("date")

loc_cal_renamed = loc_cal.rename(
    columns={
        "date": "local_holiday_date",
        "description": "next_local_holiday",
        "locale_name": "city"
    }
)

train = pd.merge_asof(
    train,
    loc_cal_renamed,
    left_on="date",
    right_on="local_holiday_date",
    by="city",
    direction="forward"
)

train["days_until_next_local_holiday"] = (
    train["local_holiday_date"] - train["date"]
).dt.days

train = train.drop(["local_holiday_date"], axis=1)

train["next_local_holiday"] = np.where(train["days_until_next_local_holiday"]>21, np.nan, train["next_local_holiday"])
train["days_until_next_local_holiday"] = np.where(train["days_until_next_local_holiday"]>21, np.nan, train["days_until_next_local_holiday"])

train["binned_days_until_next_local_holiday"] = np.where(train["days_until_next_local_holiday"]<2, 1, 
                                                            np.where(train["days_until_next_local_holiday"]<8, 2, 
                                                                     np.where(train["days_until_next_local_holiday"]<=21, 3, 4)))

train = train.drop("days_until_next_local_holiday", axis=1)

In [27]:
# National hols to test

test["date"] = pd.to_datetime(test["date"])
nat_cal["date"] = pd.to_datetime(nat_cal["date"])

nat_cal = nat_cal.sort_values("date")

nat_cal_renamed = nat_cal.rename(
    columns={
        "date": "national_holiday_date",
        "description": "next_national_holiday",
        "locale_name": "country"
    }
)

test = pd.merge_asof(
    test,
    nat_cal_renamed,
    left_on="date",
    right_on="national_holiday_date",
    direction="forward"
)

test["days_until_next_national_holiday"] = (
    test["national_holiday_date"] - test["date"]
).dt.days

test = test.drop(["national_holiday_date"], axis=1)

test["next_national_holiday"] = np.where(test["days_until_next_national_holiday"]>21, np.nan, test["next_national_holiday"])
test["days_until_next_national_holiday"] = np.where(test["days_until_next_national_holiday"]>21, np.nan, test["days_until_next_national_holiday"])

test["binned_days_until_next_national_holiday"] = np.where(test["days_until_next_national_holiday"]<2, 1, 
                                                            np.where(test["days_until_next_national_holiday"]<8, 2, 
                                                                     np.where(test["days_until_next_national_holiday"]<=21, 3, 4)))

test = test.drop(["days_until_next_national_holiday", "country"], axis=1)

In [28]:
# Regional hols to test

test["date"] = pd.to_datetime(test["date"])
reg_cal["date"] = pd.to_datetime(reg_cal["date"])

reg_cal = reg_cal.sort_values("date")

reg_cal_renamed = reg_cal.rename(
    columns={
        "date": "regional_holiday_date",
        "description": "next_regional_holiday",
        "locale_name": "state"
    }
)

test = pd.merge_asof(
    test,
    reg_cal_renamed,
    left_on="date",
    right_on="regional_holiday_date",
    by="state",
    direction="forward"
)

test["days_until_next_regional_holiday"] = (
    test["regional_holiday_date"] - test["date"]
).dt.days

test = test.drop(["regional_holiday_date"], axis=1)

test["next_regional_holiday"] = np.where(test["days_until_next_regional_holiday"]>21, np.nan, test["next_regional_holiday"])
test["days_until_next_regional_holiday"] = np.where(test["days_until_next_regional_holiday"]>21, np.nan, test["days_until_next_regional_holiday"])

test["binned_days_until_next_regional_holiday"] = np.where(test["days_until_next_regional_holiday"]<2, 1, 
                                                            np.where(test["days_until_next_regional_holiday"]<8, 2, 
                                                                     np.where(test["days_until_next_regional_holiday"]<=21, 3, 4)))

test = test.drop("days_until_next_regional_holiday", axis=1)

In [29]:
# Local hols to test

test["date"] = pd.to_datetime(test["date"])
loc_cal["date"] = pd.to_datetime(loc_cal["date"])

loc_cal = loc_cal.sort_values("date")

loc_cal_renamed = loc_cal.rename(
    columns={
        "date": "local_holiday_date",
        "description": "next_local_holiday",
        "locale_name": "city"
    }
)

test = pd.merge_asof(
    test,
    loc_cal_renamed,
    left_on="date",
    right_on="local_holiday_date",
    by="city",
    direction="forward"
)

test["days_until_next_local_holiday"] = (
    test["local_holiday_date"] - test["date"]
).dt.days

test = test.drop(["local_holiday_date"], axis=1)

test["next_local_holiday"] = np.where(test["days_until_next_local_holiday"]>21, np.nan, test["next_local_holiday"])
test["days_until_next_local_holiday"] = np.where(test["days_until_next_local_holiday"]>21, np.nan, test["days_until_next_local_holiday"])

test["binned_days_until_next_local_holiday"] = np.where(test["days_until_next_local_holiday"]<2, 1, 
                                                            np.where(test["days_until_next_local_holiday"]<8, 2, 
                                                                     np.where(test["days_until_next_local_holiday"]<=21, 3, 4)))

test = test.drop("days_until_next_local_holiday", axis=1)

In [30]:
# We did just delete earthquake data from 2016, so create eq_delta feature to capture days since earthquake and help model know which spending patterns were for earthuake relief

train["date"] = pd.to_datetime(train["date"])
ref_date = pd.Timestamp("2016-04-16")

delta = (train["date"] - ref_date).dt.days
train["eq_delta"] = delta.where((delta >= 0) & (delta <= 60), np.nan)
train["binned_eq_delta"] = (pd.cut(train["eq_delta"], bins=[-float("inf"), 1, 7, 14, 28, float("inf")], labels=[1, 2, 3, 4, 5]).cat.add_categories([0])
.fillna(0)
).astype(int)
train["binned_eq_delta"] = np.where((train["date"]>ref_date) & (train["binned_eq_delta"]==0), 6, train["binned_eq_delta"]).astype(int)

test["binned_eq_delta"] = 6


### Oil

In [31]:
oil.date.max()

'2017-08-31'

In [32]:
oil_price_dates = list(oil["date"].astype(str).unique())
needed_dates = list(train["date"].astype(str).unique())
date_diffs = [i for i in needed_dates if i not in oil_price_dates]

In [33]:
train[train["date"].astype(str).isin(date_diffs)]["day_of_week"].value_counts()

day_of_week
5    429462
6    429462
Name: count, dtype: int64

In [35]:
# Only weekends are outright missing which is a win

nas = list(oil[oil["dcoilwtico"].isna()]["date"])
train[train["date"].astype(str).isin(nas)]["day_of_week"].value_counts()

day_of_week
0    40986
4    16038
3    12474
1     3564
2     3564
Name: count, dtype: int64

In [36]:
100*(oil[oil["dcoilwtico"].isna()].shape[0]/oil.shape[0])

3.5303776683087027

In [37]:
oil['date'] = pd.to_datetime(oil['date'])
oil = oil.set_index('date')

full_range = pd.date_range(
    start=oil.index.min(),
    end=oil.index.max(),
    freq='D'
)

oil = oil.reindex(full_range)
oil.index.name = 'date'
oil = oil.sort_index()

In [38]:
oil['week_period'] = oil.index.to_period('W-SUN')

oil['week_id'] = (
    oil['week_period']
    .astype(str)
    .astype('category')
    .cat.codes + 1
)


In [39]:
weekly_avg = (
    oil.groupby('week_id')['dcoilwtico']
       .mean()
       .rename('weekly_avg_oil')
)


In [40]:
oil = oil.merge(
    weekly_avg,
    left_on='week_id',
    right_index=True,
    how='left'
).reset_index()

In [41]:
train = train.merge(oil[["date", "week_id", "weekly_avg_oil"]], on="date", how="left")
test = test.merge(oil[["date", "week_id", "weekly_avg_oil"]], on="date", how="left")

## Creation of Time-Implicit Features
### We are ignoring standard time-series methodologies here because they aren't as accurate as GBDT models.
### Therefore, we need to "tell" the models how to interpret time.
### We are now training the model with what happened sales-wise on a certain day, and telling it what happened on and before that particular day. So in test we can do the same.
### Capturing holidays & the earthquakes etc 

In [42]:
pd.set_option("display.max_columns", 28)

train.head(10)

,id,date,store_nbr,family,sales,onpromotion,day_of_week,day_of_month,week_of_year,month_of_year,christmas_flag,nyd_flag,city,state,type,cluster,next_national_holiday,binned_days_until_next_national_holiday,next_regional_holiday,binned_days_until_next_regional_holiday,next_local_holiday,binned_days_until_next_local_holiday,eq_delta,binned_eq_delta,week_id,weekly_avg_oil
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,1,1,1,1,0,1,Quito,Pichincha,D,13,Primer dia del ano,1,NaN,4,NaN,4,NaN,0,1,93.076667
1,1,2013-01-01,1,BABY CARE,0.0,0,1,1,1,1,0,1,Quito,Pichincha,D,13,Primer dia del ano,1,NaN,4,NaN,4,NaN,0,1,93.076667
2,2,2013-01-01,1,BEAUTY,0.0,0,1,1,1,1,0,1,Quito,Pichincha,D,13,Primer dia del ano,1,NaN,4,NaN,4,NaN,0,1,93.076667
3,3,2013-01-01,1,BEVERAGES,0.0,0,1,1,1,1,0,1,Quito,Pichincha,D,13,Primer dia del ano,1,NaN,4,NaN,4,NaN,0,1,93.076667
4,4,2013-01-01,1,BOOKS,0.0,0,1,1,1,1,0,1,Quito,Pichincha,D,13,Primer dia del ano,1,NaN,4,NaN,4,NaN,0,1,93.076667
5,5,2013-01-01,1,BREAD/BAKERY,0.0,0,1,1,1,1,0,1,Quito,Pichincha,D,13,Primer dia del ano,1,NaN,4,NaN,4,NaN,0,1,93.076667
6,6,2013-01-01,1,CELEBRATION,0.0,0,1,1,1,1,0,1,Quito,Pichincha,D,13,Primer dia del ano,1,NaN,4,NaN,4,NaN,0,1,93.076667
7,7,2013-01-01,1,CLEANING,0.0,0,1,1,1,1,0,1,Quito,Pichincha,D,13,Primer dia del ano,1,NaN,4,NaN,4,NaN,0,1,93.076667
8,8,2013-01-01,1,DAIRY,0.0,0,1,1,1,1,0,1,Quito,Pichincha,D,13,Primer dia del ano,1,NaN,4,NaN,4,NaN,0,1,93.076667
9,9,2013-01-01,1,DELI,0.0,0,1,1,1,1,0,1,Quito,Pichincha,D,13,Primer dia del ano,1,NaN,4,NaN,4,NaN,0,1,93.076667


In [43]:
train["set"] = "Train"

test["sales"] = np.nan
test["set"] = "Test"

cb_train_test = pd.concat([train, test], axis=0)

In [44]:
lags = [1, 2, 7, 14, 28, 56]
oil_lags = [7, 14, 28, 56]
ma_periods = [7, 28, 56]
ma_mask_flags = ["christmas_flag", "nyd_flag"]

for lag in lags:
    cb_train_test[f"sales_lag_{lag}"] = cb_train_test.groupby(["store_nbr", "family"])["sales"].shift(lag)
    cb_train_test[f"onpromotion_lag_{lag}"] = cb_train_test.groupby(["store_nbr", "family"])["onpromotion"].shift(lag) 

for lag in oil_lags:
    cb_train_test[f"oil_price_lag_{int(lag/7)}_week"] = cb_train_test.groupby(["store_nbr", "family"])["weekly_avg_oil"].shift(lag)  

for period in ma_periods:
    cb_train_test[f"sales_masked_{period}"] = cb_train_test["sales"].where(~cb_train_test[ma_mask_flags].any(axis=1))

    cb_train_test[f"sales_ma_{period}"] = (
        cb_train_test.groupby(["store_nbr", "family"])[f"sales_masked_{period}"]
        .transform(lambda x: x.shift(1).rolling(window=period, min_periods=1).mean())
    )
    cb_train_test[f"sales_std_ma_{period}"] = (
        cb_train_test.groupby(["store_nbr", "family"])[f"sales_masked_{period}"]
        .transform(lambda x: x.shift(1).rolling(window=period, min_periods=1).std())
    )
    cb_train_test[f"sales_std_ma_{period}_norm"] = cb_train_test[f"sales_std_ma_{period}"]/(cb_train_test[f"sales_ma_{period}"]+0.000001)
    cb_train_test[f"yday_sales_perc_diff_to_{period}_ma"] = (cb_train_test["sales_lag_1"]/(cb_train_test[f"sales_ma_{period}"]+0.000001))-1

    cb_train_test.drop(columns=[f"sales_masked_{period}"], inplace=True)

    cb_train_test[f"onpromotion_masked_{period}"] = cb_train_test["onpromotion"].where(~cb_train_test[ma_mask_flags].any(axis=1))

    cb_train_test[f"onpromotion_ma_{period}"] = (
        cb_train_test.groupby(["store_nbr", "family"])[f"onpromotion_masked_{period}"]
        .transform(lambda x: x.shift(1).rolling(window=period, min_periods=1).mean())
    )
    cb_train_test[f"onpromotion_std_ma_{period}"] = (
        cb_train_test.groupby(["store_nbr", "family"])[f"onpromotion_masked_{period}"]
        .transform(lambda x: x.shift(1).rolling(window=period, min_periods=1).std())
    )
    cb_train_test[f"onpromotion_std_ma_{period}_norm"] = cb_train_test[f"onpromotion_std_ma_{period}"]/(cb_train_test[f"onpromotion_ma_{period}"]+0.000001)
    cb_train_test[f"yday_onpromotion_perc_diff_to_{period}_ma"] = (cb_train_test["onpromotion_lag_1"]/(cb_train_test[f"onpromotion_ma_{period}"]+0.000001))-1

    cb_train_test.drop(columns=[f"onpromotion_masked_{period}"], inplace=True)    

## Modelling Data Prep

In [45]:
# First 56 days are used to kick off lags. Get rid so the nulls for these observations don't get mis-interpreted by the models

train = cb_train_test[cb_train_test["set"]=="Train"].reset_index(drop=True).drop("set", axis=1)
test = cb_train_test[cb_train_test["set"]=="Test"].reset_index(drop=True).drop("set", axis=1)

train = train[~train["sales_lag_56"].isna()].reset_index(drop=True)

pd.set_option("display.max_rows", 200)
pd.DataFrame(train.isna().sum())

,0
id,0
date,0
store_nbr,0
family,0
sales,0
onpromotion,0
day_of_week,0
day_of_month,0
week_of_year,0
month_of_year,0


In [46]:
# Define train, validate, and internal test sets for modelling
# These must be in chronological order to avoid look-ahead bias

print(train["date"].min())
print(train["date"].max())

# Take last calendar year and go 6 months val 6 months internal test

min_date = train["date"].min() - pd.Timedelta(days=1)
trim_date = pd.Timestamp(dt.date(2015, 7, 31))
train_end_date = pd.Timestamp(dt.date(2019, 7, 31))
val_end_date = pd.Timestamp(dt.date(2019, 8, 7))
max_date = pd.Timestamp(dt.date(2020, 8, 7))

bins = [min_date, trim_date, train_end_date, val_end_date, max_date]

labels = ["trim", "train", "val", "int_test"]

train["internal_set"] = pd.cut(
    pd.to_datetime(train["date"]),
    bins=bins,
    labels=labels,
    include_lowest=True, 
    right=True           
)

train["internal_set"].value_counts(dropna=False)

train["store_number_fam"] = train["store_nbr"].astype(str) + train["family"]

2013-02-26 00:00:00
2017-08-15 00:00:00


In [47]:
store_fam_gb = train[train["date"]>=dt.datetime(2017, 5, 1)].groupby(["store_nbr", "family"]).agg({
    "sales": "sum"
}).reset_index()

no_sales_gb = store_fam_gb[store_fam_gb["sales"]==0].reset_index()

no_sales_gb["store_nbr_fam"] = no_sales_gb["store_nbr"].astype(str) + no_sales_gb["family"]

no_sales_dict = dict(zip(no_sales_gb["store_nbr_fam"], no_sales_gb["sales"]))

train["no_sales_sales"] = train["store_number_fam"].map(no_sales_dict)

# train = train[(train["no_sales_sales"].isna()) | (train["internal_set"]=="int_test")].reset_index(drop=True).drop(["no_sales_sales", "store_number_fam"], axis=1)
train = train.drop(["no_sales_sales", "store_number_fam"], axis=1)

In [48]:
store_open_dates_gb = train[train["sales"]>0].groupby(["store_nbr"]).agg({
    "date": "min"
}).reset_index()

store_open_dates_dict = dict(zip(store_open_dates_gb["store_nbr"], store_open_dates_gb["date"]))

train["store_open_date"] = train["store_nbr"].map(store_open_dates_dict)

train = train[train["date"]>=train["store_open_date"]].reset_index(drop=True).drop("store_open_date", axis=1)

In [49]:
# Retype necessary features and declare intended categoricals, drop un-needed cols

num_to_str = ["store_nbr"]

for ds in [train, test]:
    ds[num_to_str] = ds[num_to_str].astype(str)

cats = ["store_nbr", "day_of_week",
        "month_of_year", "city", "state", 
        "type", "cluster", "next_national_holiday",
        "next_regional_holiday", "next_local_holiday"
        ]

drops = ["eq_delta"]

train = train.drop("eq_delta", axis=1)

train[cats] = train[cats].astype("category")

no_models = ["week_id", "date", "internal_set", "sales"]

In [50]:
train_X = train[train["internal_set"]=="train"].drop(no_models, axis=1)
train_y = train[train["internal_set"]=="train"]["sales"]
train_y_log1p = np.log1p(train_y).rename("l1p_sales")

w_train = pd.concat([train_X, train_y, train_y_log1p], axis=1)

val_X = train[train["internal_set"]=="val"].drop(no_models, axis=1)
val_y = train[train["internal_set"]=="val"]["sales"]
val_y_log1p = np.log1p(val_y).rename("l1p_sales")

w_val = pd.concat([val_X, val_y, val_y_log1p], axis=1)

int_test_X = train[train["internal_set"]=="int_test"].drop(no_models, axis=1)
int_test_y = train[train["internal_set"]=="int_test"]["sales"]
int_test_y_log1p = np.log1p(int_test_y).rename("l1p_sales")

w_int_test = pd.concat([int_test_X, int_test_y, int_test_y_log1p], axis=1)

In [51]:
families_list = list(train["family"].unique())

families_train_dict = {}
families_val_dict = {}
families_int_test_dict = {}

for fam in families_list:
    families_train_dict[fam] = w_train[w_train["family"]==fam]
    families_val_dict[fam] = w_val[w_val["family"]==fam]
    families_int_test_dict[fam] = w_int_test[w_int_test["family"]==fam]

In [ ]:
vol_gb = train.groupby("family").agg({
    "sales": "sum"
}).reset_index().sort_values("sales", ascending=False)

high_vol_fams = list(vol_gb[vol_gb["sales"]>10000000]["family"])
low_vol_fams = [fam for fam in families_list if fam not in high_vol_fams]

random.seed(99)  

sampled_hvf = random.sample(high_vol_fams, 3)
sampled_lvf = random.sample(low_vol_fams, 3)

In [53]:
hvf_train = w_train[w_train["family"].isin(sampled_hvf)].reset_index(drop=True)
lvf_train = w_train[w_train["family"].isin(sampled_lvf)].reset_index(drop=True)

hvf_val = w_val[w_val["family"].isin(sampled_hvf)].reset_index(drop=True)
lvf_val = w_val[w_val["family"].isin(sampled_lvf)].reset_index(drop=True)

In [54]:
hvf_m_ds = lgbm.Dataset(
    data=hvf_train.drop(["id", "family", "sales", "l1p_sales"], axis=1),
    label=hvf_train["l1p_sales"],
    categorical_feature=cats
    )

hvf_v_ds = lgbm.Dataset(
    data=hvf_val.drop(["id", "family", "sales", "l1p_sales"], axis=1),
    label=hvf_val["l1p_sales"],
    categorical_feature=cats
    )

lvf_m_ds = lgbm.Dataset(
    data=lvf_train.drop(["id", "family", "sales", "l1p_sales"], axis=1),
    label=lvf_train["l1p_sales"],
    categorical_feature=cats
    )

lvf_v_ds = lgbm.Dataset(
    data=lvf_val.drop(["id", "family", "sales", "l1p_sales"], axis=1),
    label=lvf_val["l1p_sales"],
    categorical_feature=cats
    )

In [ ]:
runthis = False

if runthis:

    param_grid = {
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 5, 7, 9, 11],
        'num_leaves': [15, 31, 63, 127],
        'num_boost_round': [100, 200, 500, 1000, 2000]
    }

    base_params = {
        'objective': 'tweedie',
        'tweedie_variance_power': 1.5,
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'verbose': -1,
        'seed': 99
    }

    # Grid search
    results = []
    best_score = float('inf')
    best_params = None
    best_model = None

    for lr, depth, leaves, rounds in itertools.product(
        param_grid['learning_rate'],
        param_grid['max_depth'],
        param_grid['num_leaves'],
        param_grid['num_boost_round']
    ):
        params = base_params.copy()
        params['learning_rate'] = lr
        params['max_depth'] = depth
        params['num_leaves'] = leaves
        
        model = lgbm.train(
            params=params,
            train_set=lvf_m_ds,
            valid_sets=[lvf_v_ds],
            num_boost_round=rounds,
            callbacks=[lgbm.early_stopping(50, verbose=False)]
        )
        
        # Evaluate on validation set with RMSLE
        val_pred_log = model.predict(lvf_val.drop(["id", "family", "sales", "l1p_sales"], axis=1))
        
        # Clip predictions to prevent overflow (exp(20) is already huge)
        val_pred_log = np.clip(val_pred_log, -10, 20)
        val_pred = np.expm1(val_pred_log)  # Convert back to original scale
        val_actual = lvf_val["sales"]  # Use original sales
        
        # Calculate RMSLE (add 1 to avoid log(0))
        val_rmsle = np.sqrt(np.mean((np.log1p(val_pred) - np.log1p(val_actual)) ** 2))
        
        results.append({
            'learning_rate': lr,
            'max_depth': depth,
            'num_leaves': leaves,
            'num_boost_round': rounds,
            'val_rmsle': val_rmsle,
            'actual_rounds': model.best_iteration
        })
        
        if val_rmsle < best_score:
            best_score = val_rmsle
            best_params = {'max_depth': depth, 'num_leaves': leaves, 'num_boost_round': rounds}
            best_model = model
        
        print(f"Learning Rate: {lr}, Depth: {depth}, Leaves: {leaves}, Rounds: {rounds} -> Val RMSLE: {val_rmsle:.4f}")

    # Show best results
    print(f"\nBest parameters: {best_params}")
    print(f"Best validation RMSLE: {best_score:.4f}")

    # # Test set evaluation with RMSLE
    # test_pred_log = best_model.predict(lvf_int_test.drop(["family", "sales", "l1p_sales"], axis=1))
    # test_pred_log = np.clip(test_pred_log, -10, 20)  # Clip here too
    # test_pred = np.expm1(test_pred_log)  # Convert back to original scale
    # test_actual = lvf_int_test["sales"]  # Use original sales
    # test_rmsle = np.sqrt(np.mean((np.log1p(test_pred) - np.log1p(test_actual)) ** 2))
    # print(f"Test RMSLE: {test_rmsle:.4f}")

In [56]:
hvf_params = {"learning_rate": 0.05, "max_depth": 5, "num_leaves": 63, "num_boost_round": 1000}
lvf_params = {"learning_rate": 0.05, 'max_depth': 9, 'num_leaves': 127, 'num_boost_round': 200}

In [ ]:
family_models_dict = {}
family_test_rmsle_dict = {}

base_params = {
    'objective': 'tweedie',
    'tweedie_variance_power': 1.5,
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'verbose': -1,
    'seed': 99
}

for fam in families_list:

    train_set = families_train_dict[fam]

    train_lgbm_set = lgbm.Dataset(data=train_set.drop(["id", "family", "sales", "l1p_sales"], axis=1),
    label=train_set["l1p_sales"],
    categorical_feature=cats)

    if fam in high_vol_fams:
        model_params = base_params.copy()
        model_params["learning_rate"] = hvf_params["learning_rate"]
        model_params["max_depth"] = hvf_params["max_depth"]
        model_params["num_leaves"] = hvf_params["num_leaves"]
        model_params["num_boost_round"] = hvf_params["num_boost_round"]
    else:
        model_params = base_params.copy()
        model_params["learning_rate"] = lvf_params["learning_rate"]
        model_params["max_depth"] = lvf_params["max_depth"]
        model_params["num_leaves"] = lvf_params["num_leaves"]
        model_params["num_boost_round"] = lvf_params["num_boost_round"] 
    
    model = lgbm.train(
    params=model_params,
    train_set=train_lgbm_set,
    )


    family_models_dict[fam] = model

In [58]:
test_dates_list = list(test["date"].unique())

train["set"] = "Train"
train = train.drop("internal_set", axis=1)
test["set"] = "Test"
test = test.drop("eq_delta", axis=1)

cb_train_test = pd.concat([train, test], axis=0)
cb_train_test[cats] = cb_train_test[cats].astype("category")

/var/folders/6c/psx92cbd57gd1gk2h4klzk_w0000gn/T/ipykernel_18768/1178277407.py:8: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cb_train_test = pd.concat([train, test], axis=0)
/var/folders/6c/psx92cbd57gd1gk2h4klzk_w0000gn/T/ipykernel_18768/1178277407.py:8: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cb_train_test = pd.concat([train, test], axis=0)


In [59]:
lags = [1, 2, 7, 14, 28, 56]
ma_periods = [7, 28, 56]


for index, date in enumerate(test_dates_list):

    if index > 0:
        cb_train_test_chunking = cb_train_test.copy()
        for lag in lags:
            cb_train_test_chunking[f"sales_lag_{lag}"] = cb_train_test_chunking.groupby(["store_nbr", "family"], observed=False)["sales"].shift(lag).fillna(0)

        for period in ma_periods:
            cb_train_test_chunking[f"sales_ma_{period}"] = (
                cb_train_test_chunking.groupby(["store_nbr", "family"], observed=False)["sales"]
                .transform(lambda x: x.shift(1).rolling(window=period, min_periods=1).mean())
            ).fillna(0)
            cb_train_test_chunking[f"sales_std_ma_{period}"] = (
                cb_train_test_chunking.groupby(["store_nbr", "family"], observed=False)["sales"]
                .transform(lambda x: x.shift(1).rolling(window=period, min_periods=1).std())
            ).fillna(0)
            cb_train_test_chunking[f"sales_std_ma_{period}_norm"] = (
                cb_train_test_chunking[f"sales_std_ma_{period}"] / (cb_train_test_chunking[f"sales_ma_{period}"] + 0.000001)
            ).fillna(0)
            cb_train_test_chunking[f"yday_sales_perc_diff_to_{period}_ma"] = ((
                cb_train_test_chunking["sales_lag_1"] / (cb_train_test_chunking[f"sales_ma_{period}"] + 0.000001)
            ) - 1).fillna(0)
        
        feature_cols = [f"sales_lag_{lag}" for lag in lags]
        for period in ma_periods:
            feature_cols.extend([
                f"sales_ma_{period}",
                f"sales_std_ma_{period}",
                f"sales_std_ma_{period}_norm",
                f"yday_sales_perc_diff_to_{period}_ma"
            ])

        cb_train_test.loc[date_mask, feature_cols] = cb_train_test_chunking.loc[date_mask, feature_cols].values

    date_mask = cb_train_test["date"] == date
    date_chunk = cb_train_test[date_mask].reset_index(drop=True)

    family_data_chunks_dict = {}
    for fam in families_list:
        fam_chunk = date_chunk[date_chunk["family"]==fam].reset_index(drop=True)
        fam_chunk_model = fam_chunk.drop(["id", "date", "family", "sales", "week_id", "set"], axis=1)
        fam_model = family_models_dict[fam]
        fam_chunk["pred_sales"] = np.round(np.expm1(fam_model.predict(fam_chunk_model)), 2)
        family_data_chunks_dict[fam] = fam_chunk[["id", "pred_sales"]]

    family_data_df = pd.concat(family_data_chunks_dict.values(), ignore_index=True).sort_values("id").reset_index(drop=True)

    cb_train_test.loc[date_mask, "sales"] = family_data_df["pred_sales"].values
    

In [60]:
lags = [1, 2, 7, 14, 28, 56]
ma_periods = [7, 28, 56]

for index, date in enumerate(test_dates_list):
    if index > 0:
        date_mask = cb_train_test["date"] == date
        
        # Compute features directly on cb_train_test (not a copy)
        # This way they use the updated sales values from previous iterations
        for lag in lags:
            cb_train_test[f"sales_lag_{lag}"] = cb_train_test.groupby(["store_nbr", "family"], observed=False)["sales"].shift(lag).fillna(0)
        
        for period in ma_periods:
            cb_train_test[f"sales_ma_{period}"] = (
                cb_train_test.groupby(["store_nbr", "family"], observed=False)["sales"]
                .transform(lambda x: x.shift(1).rolling(window=period, min_periods=1).mean())
            ).fillna(0)
            cb_train_test[f"sales_std_ma_{period}"] = (
                cb_train_test.groupby(["store_nbr", "family"], observed=False)["sales"]
                .transform(lambda x: x.shift(1).rolling(window=period, min_periods=1).std())
            ).fillna(0)
            cb_train_test[f"sales_std_ma_{period}_norm"] = (
                cb_train_test[f"sales_std_ma_{period}"] / (cb_train_test[f"sales_ma_{period}"] + 0.000001)
            ).fillna(0)
            cb_train_test[f"yday_sales_perc_diff_to_{period}_ma"] = ((
                cb_train_test["sales_lag_1"] / (cb_train_test[f"sales_ma_{period}"] + 0.000001)
            ) - 1).fillna(0)
        
        # Now work with the current date's data
        date_chunk = cb_train_test[date_mask].reset_index(drop=True)
        family_data_chunks_dict = {}
        
        for fam in families_list:
            fam_chunk = date_chunk[date_chunk["family"] == fam].reset_index(drop=True)
            fam_chunk_model = fam_chunk.drop(["id", "date", "family", "sales", "week_id", "set"], axis=1)
            fam_model = family_models_dict[fam]
            fam_chunk["pred_sales"] = np.round(np.expm1(fam_model.predict(fam_chunk_model)), 2)
            family_data_chunks_dict[fam] = fam_chunk[["id", "pred_sales"]]
        
        family_data_df = pd.concat(family_data_chunks_dict.values(), ignore_index=True).sort_values("id").reset_index(drop=True)
        cb_train_test.loc[date_mask, "sales"] = family_data_df["pred_sales"].values

In [61]:
test_finito = cb_train_test[cb_train_test["set"]=="Test"]

test_finito[["id", "sales"]].to_csv("/Users/adamgolledge/Documents/PersonalProjects/Store Sales/Submissions/plsbgood.csv", index=False)